In [0]:

from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "retail_analytics"

movies = spark.table(
    f"{CATALOG}.{SCHEMA}.silver_movies"
)

posters = spark.table(
    f"{CATALOG}.{SCHEMA}.silver_posters"
)

print("Películas Silver:", movies.count())
print("Pósteres Silver:", posters.count())


Películas Silver: 9718
Pósteres Silver: 9718


In [0]:

# ============================================================
# GOLD - DATASET ANALITICO DE PELICULAS
# ============================================================

movies_enriched = movies.join(
    posters,
    on="id",
    how="left"
)

gold_movies = (
    movies_enriched
    .withColumn(
        "release_year",
        F.year(F.col("release_date"))
    )
    .withColumn(
        "profit_usd",
        F.when(
            F.col("revenue_usd").isNotNull()
            & F.col("budget_usd").isNotNull(),
            F.col("revenue_usd") - F.col("budget_usd")
        )
    )
    .withColumn(
        "roi_pct",
        F.when(
            F.col("budget_usd") > 0,
            F.round(
                (
                    (F.col("revenue_usd") - F.col("budget_usd"))
                    / F.col("budget_usd")
                ) * 100,
                2
            )
        )
    )
    .withColumn(
        "vote_count",
        F.coalesce(F.col("vote_count"), F.lit(0))
    )
)

display(
    gold_movies.orderBy(
        F.desc("user_score"),
        F.desc("vote_count")
    )
)


id,title,genres,language,user_score,runtime_hour,runtime_min,release_date,vote_count,director,top_billed,budget_usd,revenue_usd,runtime_minutes,poster_path,backdrop_path,release_year,profit_usd,roi_pct
1,The Shawshank Redemption,"Drama, Crime",en,8.7,2,22,1994-09-23,27070,Frank Darabont,"Tim Robbins, Morgan Freeman, Bob Gunton, William Sadler, Clancy Brown",2.5E7,2.8341469E7,142,https://image.tmdb.org/t/p/w500/9cqNxx0GxF0bflZmeSMuL5tnGzr.jpg,https://image.tmdb.org/t/p/w500/zfbjgQE1uSd9wiPTX4VzsLi0rGG.jpg,1994,3341469.0,13.37
2,The Godfather,"Drama, Crime",en,8.7,2,55,1972-03-14,20563,Francis Ford Coppola,"Marlon Brando, Al Pacino, James Caan, Robert Duvall, Richard S. Castellano",6000000.0,2.45066411E8,175,https://image.tmdb.org/t/p/w500/3bhkrj58Vtu7enYsRolD1fZdja1.jpg,https://image.tmdb.org/t/p/w500/tmU7GeKVybMWFButWEGl2M4GeiP.jpg,1972,2.39066411E8,3984.44
4,Schindler's List,"Drama, History, War",en,8.6,3,15,1993-12-15,15810,Steven Spielberg,"Liam Neeson, Ben Kingsley, Ralph Fiennes, Caroline Goodall, Jonathan Sagall",2.2E7,3.21365567E8,195,https://image.tmdb.org/t/p/w500/sF1U4EUQS8YHUYjNl3pMGNIQyr0.jpg,https://image.tmdb.org/t/p/w500/zb6fM1CX41D9rF9hdgclu0peUmy.jpg,1993,2.99365567E8,1360.75
3,The Godfather Part II,"Drama, Crime",en,8.6,3,22,1974-12-20,12403,Francis Ford Coppola,"Al Pacino, Robert Duvall, Diane Keaton, Robert De Niro, John Cazale",1.3E7,1.026E8,202,https://image.tmdb.org/t/p/w500/hek3koDUyRQk7FIhPXsa6mT2Zc3.jpg,https://image.tmdb.org/t/p/w500/kGzFbGhp99zva6oZODW5atUtnqi.jpg,1974,8.96E7,689.23
9,The Dark Knight,"Drama, Action, Crime, Thriller",en,8.5,2,32,2008-07-16,32816,Christopher Nolan,"Christian Bale, Heath Ledger, Aaron Eckhart, Michael Caine, Maggie Gyllenhaal",1.85E8,1.004558444E9,152,https://image.tmdb.org/t/p/w500/qJ2tW6WMUDux911r6m7haRef0WH.jpg,https://image.tmdb.org/t/p/w500/oOv2oUXcAaNXakRqUPxYq5lJURz.jpg,2008,8.19558444E8,443.0
12,Pulp Fiction,"Thriller, Crime",en,8.5,2,34,1994-09-10,27853,Quentin Tarantino,"John Travolta, Samuel L. Jackson, Uma Thurman, Bruce Willis, Ving Rhames",8500000.0,2.13928762E8,154,https://image.tmdb.org/t/p/w500/d5iIlFn5s0ImszYzBPb8JPIfbXD.jpg,https://image.tmdb.org/t/p/w500/suaEOtk1N1sgg2MTM7oZd2cfVp3.jpg,1994,2.05428762E8,2416.81
15,Forrest Gump,"Comedy, Drama, Romance",en,8.5,2,22,1994-06-23,27384,Robert Zemeckis,"Tom Hanks, Robin Wright, Gary Sinise, Sally Field, Mykelti Williamson",5.5E7,6.77387716E8,142,https://image.tmdb.org/t/p/w500/arw2vcBveWOVZr6pxd9XTd1TdQa.jpg,https://image.tmdb.org/t/p/w500/ghgfzbEV7kbpbi1O8eIILKVXEA8.jpg,1994,6.22387716E8,1131.61
14,The Lord of the Rings: The Return of the King,"Adventure, Fantasy, Action",en,8.5,3,21,2003-12-17,24151,Peter Jackson,"Elijah Wood, Ian McKellen, Viggo Mortensen, Sean Astin, Andy Serkis",9.4E7,1.118888979E9,201,https://image.tmdb.org/t/p/w500/rCzpDGLbOoPwLjy3OAm5NUPOTrC.jpg,https://image.tmdb.org/t/p/w500/5JrZAtyk3LwiiAWLW0kwz41XZJC.jpg,2003,1.024888979E9,1090.31
11,Parasite,"Comedy, Thriller, Drama",ko,8.5,2,13,2019-05-30,18215,Bong Joon-ho,"Song Kang-ho, Lee Sun-kyun, Cho Yeo-jeong, Choi Woo-shik, Park So-dam",1.1363E7,2.57591776E8,133,https://image.tmdb.org/t/p/w500/7IiTTgloJzvGI1TAYymCfbfl3vT.jpg,https://image.tmdb.org/t/p/w500/hiKmpZMGZsrkA3cdce8a7Dpos1j.jpg,2019,2.46228776E8,2166.93
10,The Green Mile,"Fantasy, Drama, Crime",en,8.5,3,9,1999-12-10,17376,Frank Darabont,"Tom Hanks, David Morse, Bonnie Hunt, Michael Clarke Duncan, James Cromwell",6.0E7,2.86801374E8,189,https://image.tmdb.org/t/p/w500/8VG8fDNiy50H4FedGwdSVUPoaJe.jpg,https://image.tmdb.org/t/p/w500/vxJ08SvwomfKbpboCWynC3uqUg4.jpg,1999,2.26801374E8,378.0


In [0]:

# ============================================================
# GOLD - GUARDAR TABLA PRINCIPAL
# ============================================================

(
    gold_movies.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.gold_movies")
)

gold_saved = spark.table(
    f"{CATALOG}.{SCHEMA}.gold_movies"
)

print("Películas Gold:", gold_saved.count())

display(
    gold_saved.select(
        "id",
        "title",
        "genres",
        "user_score",
        "vote_count",
        "release_year",
        "budget_usd",
        "revenue_usd",
        "profit_usd",
        "roi_pct"
    ).limit(10)
)


Películas Gold: 9718


id,title,genres,user_score,vote_count,release_year,budget_usd,revenue_usd,profit_usd,roi_pct
28,One Flew Over the Cuckoo's Nest,Drama,8.4,10402,1975,3000000.0,1.08981275E8,1.05981275E8,3532.71
29,Once Upon a Time in America,"Drama, Crime",8.4,5368,1984,3.0E7,5500000.0,-2.45E7,-81.67
30,Gabriel's Inferno,"Romance, Drama",8.4,2410,2020,null,null,null,null
33,Primal: Tales of Savagery,"Action, Adventure, Animation, Drama",8.4,315,2019,null,null,null,null
93,Evangelion: 3.0+1.0 Thrice Upon a Time,"Animation, Action, Science Fiction, Drama",8.2,806,2021,2.97E7,9.3882453E7,6.4182453E7,216.1
151,8½,Drama,8.2,2260,1963,null,null,null,null
171,My Friends,Comedy,8.1,691,1975,null,null,null,null
200,All About Eve,Drama,8.1,1516,1950,1400000.0,8400000.0,7000000.0,500.0
208,WALL·E,"Animation, Family, Science Fiction",8.1,18690,2008,1.8E8,5.2131186E8,3.4131186E8,189.62
221,There Will Be Blood,Drama,8.1,6563,2007,2.5E7,7.7208711E7,5.2208711E7,208.83


In [0]:

# ============================================================
# GOLD - INDICADORES POR GENERO
# ============================================================

gold_by_genre = (
    gold_saved
    .withColumn(
        "genre",
        F.explode(
            F.split(F.coalesce(F.col("genres"), F.lit("")), r",\s*")
        )
    )
    .filter(F.col("genre") != "")
    .groupBy("genre")
    .agg(
        F.countDistinct("id").alias("movies_count"),
        F.round(F.avg("user_score"), 2).alias("avg_user_score"),
        F.sum("vote_count").alias("total_votes"),
        F.round(F.avg("budget_usd"), 2).alias("avg_budget_usd"),
        F.round(F.avg("revenue_usd"), 2).alias("avg_revenue_usd"),
        F.round(F.avg("profit_usd"), 2).alias("avg_profit_usd")
    )
    .orderBy(F.desc("movies_count"))
)

display(gold_by_genre)

(
    gold_by_genre.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.gold_genres")
)

print(
    "Géneros Gold:",
    spark.table(f"{CATALOG}.{SCHEMA}.gold_genres").count()
)


genre,movies_count,avg_user_score,total_votes,avg_budget_usd,avg_revenue_usd,avg_profit_usd
Drama,4309,6.89,7411301,2.456604695E7,5.887420579E7,4.341020587E7
Comedy,3526,6.5,5744336,3.21382863E7,9.142795248E7,6.952626809E7
Thriller,2593,6.44,4964988,2.995676981E7,7.259570219E7,5.104484729E7
Action,2240,6.5,6475499,5.665960574E7,1.5653881918E8,1.1208218161E8
Romance,1643,6.72,2596388,2.520550678E7,7.218575346E7,5.689396712E7
Adventure,1625,6.61,5679409,6.91719968E7,2.1562356635E8,1.584185929E8
Crime,1464,6.67,2876362,2.885598029E7,6.577282825E7,4.269710403E7
Horror,1427,6.18,2154563,1.559058961E7,4.632291692E7,3.700298408E7
Science Fiction,1192,6.46,3898754,5.641088475E7,1.6685570742E8,1.2535638927E8
Fantasy,1109,6.65,3028576,5.657679847E7,1.6288970702E8,1.2311866716E8


Géneros Gold: 19


In [0]:

# Validación básica de las tablas Gold

gold_movies = spark.table(
    "workspace.retail_analytics.gold_movies"
)

gold_genres = spark.table(
    "workspace.retail_analytics.gold_genres"
)

print("=== GOLD MOVIES ===")
print("Registros:", gold_movies.count())
print("IDs distintos:", gold_movies.select("id").distinct().count())
print("Títulos nulos:", gold_movies.filter("title IS NULL").count())
print("Puntuaciones nulas:", gold_movies.filter("user_score IS NULL").count())
print("Presupuestos nulos:", gold_movies.filter("budget_usd IS NULL").count())
print("Recaudaciones nulas:", gold_movies.filter("revenue_usd IS NULL").count())

print("=== GOLD GENRES ===")
print("Géneros:", gold_genres.count())

display(gold_genres.orderBy("movies_count", ascending=False))


=== GOLD MOVIES ===
Registros: 9718
IDs distintos: 9718
Títulos nulos: 0
Puntuaciones nulas: 0
Presupuestos nulos: 2983
Recaudaciones nulas: 2542
=== GOLD GENRES ===
Géneros: 19


genre,movies_count,avg_user_score,total_votes,avg_budget_usd,avg_revenue_usd,avg_profit_usd
Drama,4309,6.89,7411301,2.456604695E7,5.887420579E7,4.341020587E7
Comedy,3526,6.5,5744336,3.21382863E7,9.142795248E7,6.952626809E7
Thriller,2593,6.44,4964988,2.995676981E7,7.259570219E7,5.104484729E7
Action,2240,6.5,6475499,5.665960574E7,1.5653881918E8,1.1208218161E8
Romance,1643,6.72,2596388,2.520550678E7,7.218575346E7,5.689396712E7
Adventure,1625,6.61,5679409,6.91719968E7,2.1562356635E8,1.584185929E8
Crime,1464,6.67,2876362,2.885598029E7,6.577282825E7,4.269710403E7
Horror,1427,6.18,2154563,1.559058961E7,4.632291692E7,3.700298408E7
Science Fiction,1192,6.46,3898754,5.641088475E7,1.6685570742E8,1.2535638927E8
Fantasy,1109,6.65,3028576,5.657679847E7,1.6288970702E8,1.2311866716E8


In [0]:

from pyspark.sql import functions as F

gold = spark.table("workspace.retail_analytics.gold_movies")

print("Presupuestos negativos:")
print(gold.filter(F.col("budget_usd") < 0).count())

print("Recaudaciones negativas:")
print(gold.filter(F.col("revenue_usd") < 0).count())

print("Películas con presupuesto cero:")
print(gold.filter(F.col("budget_usd") == 0).count())

print("Películas con recaudación cero:")
print(gold.filter(F.col("revenue_usd") == 0).count())

print("ROI no nulo:")
print(gold.filter(F.col("roi_pct").isNotNull()).count())

display(
    gold.select(
        "title", "budget_usd", "revenue_usd",
        "profit_usd", "roi_pct"
    ).orderBy(F.desc("roi_pct")).limit(10)
)


Presupuestos negativos:
0
Recaudaciones negativas:
0
Películas con presupuesto cero:
0
Películas con recaudación cero:
0
ROI no nulo:
6127


title,budget_usd,revenue_usd,profit_usd,roi_pct
Paranormal Activity,15000.0,1.933558E8,1.933408E8,1288938.67
The Blair Witch Project,60000.0,2.48639099E8,2.48579099E8,414298.5
Fist of Fury,100000.0,1.0E8,9.99E7,99900.0
One Cut of the Dead,52406.0,2.759018E7,2.7537774E7,52546.99
Enter the Dragon,850000.0,4.0E8,3.9915E8,46958.82
The Gallows,100000.0,4.296441E7,4.286441E7,42864.41
Open Water,130000.0,5.47E7,5.457E7,41976.92
Bambi,858000.0,2.6744715E8,2.6658915E8,31071.0
Mad Max,350000.0,1.0E8,9.965E7,28471.43
El Mariachi,7225.0,2000000.0,1992775.0,27581.66
